# Clase 7: Criterios de Factibilidad y Pertinencia

### Machine Learning (RIA-805) — Bloque 2 — Factibilidad Técnico-Financiera

**Docente:** Marco Capio  
**Fecha:** Semana 4

**Objetivo:** Evaluar la factibilidad técnica y la pertinencia de modelos de Machine Learning antes de llevarlos a producción, aplicando criterios de rendimiento, latencia, interpretabilidad, costo y escalabilidad.

**Conexión con la práctica:** En P2 aplicarán estos criterios a su propio modelo de Wall-Following y aprenderán a traducir errores a costos en dólares.

## ¿Dónde estamos?

En el **Bloque 1** construimos un pipeline de Machine Learning: exploramos los datos (EDA), preprocesamos variables, entrenamos modelos supervisados y no supervisados, y comparamos modelos mediante benchmarking. También dejamos artefactos reutilizables, como modelos, escalador y codificador de etiquetas.

> En el Bloque 1 respondimos: **¿funciona el modelo?** En el Bloque 2 responderemos: **¿vale la pena llevarlo a producción?**

Una métrica alta es importante, pero por sí sola no garantiza que la solución sea viable, útil para las personas usuarias o sostenible en su entorno.

## ¿Qué es factibilidad técnica?

La **factibilidad técnica** determina si una solución puede construirse, desplegarse y operarse de forma confiable con los datos, la infraestructura, el personal y las restricciones disponibles. No es solo comprobar que el algoritmo entrena: también se evalúan su integración, velocidad, consumo de recursos, seguridad y capacidad de mantenimiento.

**Preguntas clave:** ¿Tenemos suficientes datos de calidad? ¿El modelo cabe en el dispositivo o servidor disponible? ¿Responde dentro del tiempo requerido? ¿Se integra con los sistemas existentes? ¿Podemos monitorearlo y reentrenarlo cuando cambien los datos?

**Ejemplo:** un robot móvil debe reaccionar a obstáculos en tiempo real. Un modelo con muy buen rendimiento promedio, pero que tarda varios segundos por predicción, no sería técnicamente factible para evitar una colisión. La latencia debe compararse con el plazo de respuesta seguro del robot.

## ¿Qué es pertinencia?

La **pertinencia** evalúa si el problema elegido, los datos, las predicciones y sus consecuencias responden a una necesidad real de las personas u organización. Un modelo puede ser técnicamente correcto y, aun así, no ser pertinente si optimiza una métrica equivocada, perjudica a ciertos grupos o no mejora la decisión que se quería apoyar.

**Preguntas clave:** ¿La predicción resuelve una necesidad concreta? ¿Qué errores son más costosos? ¿La métrica representa el objetivo del negocio? ¿Quién recibe el beneficio y quién asume el riesgo? ¿Se puede actuar a partir de la predicción?

**Ejemplo:** en una predicción de churn (abandono de clientes), una alta *accuracy* puede ocultar que el modelo no detecta a quienes realmente abandonarán. Si la empresa quiere contactarlos a tiempo, el *recall* de la clase positiva puede ser más pertinente: los falsos negativos representan clientes que se pierden sin intervención. La elección de métrica depende del costo de cada tipo de error.

## Criterios de selección de modelos

Al comparar alternativas, consideraremos seis criterios complementarios:

1. **Rendimiento:** calidad de las predicciones con métricas adecuadas al problema y a sus errores.
2. **Latencia:** tiempo que transcurre desde que se solicita una predicción hasta que se obtiene.
3. **Interpretabilidad:** facilidad para explicar qué factores influyen en una predicción y cómo se llegó a ella.
4. **Costo:** recursos y dinero necesarios para entrenar, alojar y ejecutar el modelo, incluidos RAM, CPU/GPU y energía.
5. **Escalabilidad:** capacidad para responder al aumento de datos, usuarios o solicitudes sin degradar el servicio de forma inaceptable.
6. **Mantenibilidad:** facilidad para probar, documentar, monitorear, actualizar y operar el modelo a lo largo del tiempo.

Las mediciones de este notebook son una primera comparación en un entorno local; no sustituyen pruebas de carga ni una evaluación de producción.

In [ ]:
# Importamos las herramientas de datos, visualización, persistencia y evaluación.
import os
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

# Activamos la visualización de gráficos dentro del notebook.
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Localizamos la raíz del proyecto aunque Jupyter se haya iniciado en notebooks/.
candidatos_raiz = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (ruta for ruta in candidatos_raiz if (ruta / 'data' / 'processed' / 'crop.parquet').is_file()),
    Path.cwd(),
)
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'crop.parquet'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print(f'Raíz del proyecto: {PROJECT_ROOT}')
print(f'Versión de scikit-learn: {sklearn.__version__}')

## Carga del pipeline del Bloque 1

Vamos a reutilizar el pipeline que construimos en el Bloque 1. Cargamos los modelos entrenados, el *scaler* y el *label encoder* desde `models/`, además del dataset procesado `data/processed/crop.parquet`.

La comparación usa los mismos datos y la misma partición de entrenamiento/prueba del Bloque 1. Si alguno de los modelos no está disponible o no puede cargarse, se mostrará una advertencia y continuaremos con los artefactos utilizables.

In [ ]:
# Cargamos el dataset requerido; sin él no es posible evaluar los modelos.
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f'No se encontró {DATA_PATH}. Verifica que el dataset procesado del Bloque 1 exista.'
    )
df = pd.read_parquet(DATA_PATH)
print(f'Dataset cargado: {df.shape[0]} filas y {df.shape[1]} columnas.')

# Cargamos cada modelo de manera independiente para poder continuar si falta alguno.
rutas_modelos = {
    'Random Forest': MODELS_DIR / 'random_forest.pkl',
    'XGBoost': MODELS_DIR / 'xgboost.pkl',
    'SVM': MODELS_DIR / 'svm.pkl',
}
modelos = {}
for nombre, ruta in rutas_modelos.items():
    if not ruta.is_file():
        print(f'Advertencia: no existe el modelo {nombre}: {ruta}')
        continue
    try:
        modelos[nombre] = joblib.load(ruta)
        print(f'Modelo cargado: {nombre}')
    except Exception as error:
        print(f'Advertencia: no se pudo cargar {nombre} ({type(error).__name__}: {error})')

# El escalador y el codificador son necesarios para preparar los datos como en el Bloque 1.
artefactos = {}
for nombre, ruta in {
    'scaler': MODELS_DIR / 'scaler.pkl',
    'label_encoder': MODELS_DIR / 'label_encoder.pkl',
}.items():
    if not ruta.is_file():
        print(f'Advertencia: no existe el artefacto {nombre}: {ruta}')
        continue
    try:
        artefactos[nombre] = joblib.load(ruta)
        print(f'Artefacto cargado: {nombre}')
    except Exception as error:
        print(f'Advertencia: no se pudo cargar {nombre} ({type(error).__name__}: {error})')

scaler = artefactos.get('scaler')
label_encoder = artefactos.get('label_encoder')
if label_encoder is not None:
    print(f'Clases ({len(label_encoder.classes_)}): {list(label_encoder.classes_)}')
elif 'label' in df.columns:
    print(f'Clases en el dataset: {sorted(df["label"].dropna().unique())}')

## Criterio 1: Rendimiento

El **rendimiento** describe qué tan bien predice el modelo, pero la métrica correcta depende del problema:

- **Accuracy:** proporción total de predicciones correctas.
- **Precision:** de las predicciones positivas, qué proporción fue correcta.
- **Recall:** de los positivos reales, qué proporción detectó el modelo.
- **F1:** media armónica de *precision* y *recall*; resume su equilibrio.
- **ROC-AUC:** capacidad de ordenar casos positivos por encima de negativos a través de umbrales. En clasificación multiclase se estima aquí con estrategia *one-vs-rest*.

Como Crop Recommendation tiene varias clases, reportaremos *precision*, *recall* y F1 ponderados por el soporte de cada clase. Vamos a calcular las métricas de cada modelo para comparar rendimiento.

In [ ]:
# Preparamos los datos con los artefactos ya entrenados y el mismo split del Bloque 1.
resultados = pd.DataFrame(
    columns=['modelo', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']
)
X_test_scaled = pd.DataFrame()
y_test = np.array([])

if scaler is None or label_encoder is None:
    print('Advertencia: no se calculan métricas porque falta el scaler o el label encoder.')
elif 'label' not in df.columns:
    print("Advertencia: no se encontró la columna objetivo 'label' en el dataset.")
else:
    X = df.drop(columns=['label']).copy()
    y = label_encoder.transform(df['label'])
    X_scaled = pd.DataFrame(
        scaler.transform(X), columns=X.columns, index=X.index
    )
    _, X_test_scaled, _, y_test = train_test_split(
        X_scaled,
        y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    filas_metricas = []
    for nombre, modelo in modelos.items():
        predicciones = modelo.predict(X_test_scaled)
        fila = {
            'modelo': nombre,
            'accuracy': accuracy_score(y_test, predicciones),
            'precision': precision_score(y_test, predicciones, average='weighted', zero_division=0),
            'recall': recall_score(y_test, predicciones, average='weighted', zero_division=0),
            'f1': f1_score(y_test, predicciones, average='weighted', zero_division=0),
            'roc_auc': np.nan,
        }
        # ROC-AUC requiere puntuaciones por clase y puede no estar soportado por todos los estimadores.
        try:
            if hasattr(modelo, 'predict_proba'):
                puntuaciones = modelo.predict_proba(X_test_scaled)
            elif hasattr(modelo, 'decision_function'):
                puntuaciones = modelo.decision_function(X_test_scaled)
            else:
                puntuaciones = None
            if puntuaciones is not None:
                fila['roc_auc'] = roc_auc_score(
                    y_test,
                    puntuaciones,
                    labels=getattr(modelo, 'classes_', None),
                    multi_class='ovr',
                    average='weighted',
                )
            else:
                print(f'Advertencia: {nombre} no ofrece puntuaciones para calcular ROC-AUC.')
        except (ValueError, TypeError) as error:
            print(f'Advertencia: ROC-AUC no disponible para {nombre}: {error}')
        filas_metricas.append(fila)

    resultados = pd.DataFrame(filas_metricas, columns=resultados.columns)

if resultados.empty:
    print('No hay modelos disponibles para calcular las métricas.')
else:
    print(f'Test: {len(X_test_scaled)} observaciones; {len(np.unique(y_test))} clases.')
    display(resultados.round(4))

## Criterio 2: Latencia

La **latencia** es el tiempo que tarda el sistema en responder a una solicitud de predicción. Se mide con un reloj de alta resolución alrededor de `predict`, idealmente después de una predicción de calentamiento. En este ejercicio reportamos el promedio de predicciones individuales en milisegundos.

La latencia importa en aplicaciones interactivas y de tiempo real: un sistema puede ser preciso, pero inservible si responde después de que la decisión debía tomarse. El resultado depende del hardware, del tamaño de entrada, de la carga concurrente y del software.

Vamos a medir la latencia de inferencia de cada modelo con 100 predicciones (o todas las disponibles si el conjunto de prueba es menor).

In [ ]:
def medir_latencia(modelo, X_datos, n=100):
    """Devuelve la latencia media, en milisegundos, de n predicciones individuales."""
    if n <= 0:
        raise ValueError('n debe ser un entero positivo.')
    if len(X_datos) == 0:
        raise ValueError('X_datos no puede estar vacío.')

    # Calentamos el estimador y limitamos la medición al número de filas disponibles.
    modelo.predict(X_datos.iloc[[0]])
    cantidad = min(n, len(X_datos))
    duraciones = []
    for indice in range(cantidad):
        muestra = X_datos.iloc[[indice]]
        inicio = time.perf_counter()
        modelo.predict(muestra)
        duraciones.append(time.perf_counter() - inicio)
    return float(np.mean(duraciones) * 1000)


# Medimos la misma operación individual para cada modelo evaluado.
latencias = {}
if len(X_test_scaled) == 0:
    print('Advertencia: no se mide latencia porque no hay datos de prueba preparados.')
else:
    for nombre, modelo in modelos.items():
        if nombre in resultados['modelo'].values:
            latencias[nombre] = medir_latencia(modelo, X_test_scaled, n=100)
    resultados['latencia_ms'] = resultados['modelo'].map(latencias)
    display(resultados.round(4))

## Criterio 3: Interpretabilidad

La **interpretabilidad** es la facilidad con que las personas pueden entender las razones de una predicción. Puede ser importante para auditar decisiones, comunicar resultados, detectar errores y generar confianza. La explicación disponible depende del modelo y del método empleado; una explicación de importancia global no justifica por sí sola una predicción individual.

Random Forest permite consultar importancias de variables y XGBoost ofrece herramientas similares. Un SVM con kernel no lineal suele ser menos interpretable directamente; las redes neuronales también suelen requerir técnicas adicionales. Estas son tendencias, no garantías: una importancia global no demuestra causalidad.

Random Forest permite ver importancia de features. XGBoost también. SVM y redes neuronales no suelen ofrecer una explicación directa equivalente.

In [ ]:
# Graficamos la importancia global de variables si el Random Forest está disponible.
if 'Random Forest' not in modelos:
    print('Advertencia: no se genera la gráfica porque Random Forest no está disponible.')
elif not hasattr(modelos['Random Forest'], 'feature_importances_'):
    print('Advertencia: el modelo Random Forest no expone feature_importances_.')
else:
    importancias = pd.Series(
        modelos['Random Forest'].feature_importances_,
        index=df.drop(columns=['label']).columns,
        name='Importancia',
    ).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(x=importancias.values, y=importancias.index, color='seagreen', ax=ax)
    ax.set_title('Importancia global de variables — Random Forest')
    ax.set_xlabel('Importancia relativa')
    ax.set_ylabel('Variable')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'importancia_features.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Variables más relevantes según Random Forest:')
    display(importancias.head(7).to_frame().round(4))
    print(f'Gráfico guardado en: {FIGURES_DIR / "importancia_features.png"}')

## Criterio 4: Costo computacional

El **costo computacional** incluye los recursos requeridos durante entrenamiento e inferencia. Un modelo puede necesitar más **RAM** para cargarse, más **CPU/GPU** para procesar solicitudes y más energía cuando se ejecuta continuamente. Estos recursos se traducen en costos de infraestructura y operación.

El tamaño del archivo serializado es solo una aproximación: no equivale exactamente a la memoria RAM ocupada ni al costo energético. Aun así, comparar el tamaño de los artefactos ayuda a identificar diferencias que conviene investigar.

Vamos a medir el tamaño de cada modelo en disco y estimar el uso de memoria mediante ese tamaño como referencia inicial.

In [ ]:
# Medimos los artefactos serializados; un modelo faltante se representa como dato no disponible.
tamanos_kb = {}
for nombre, ruta in rutas_modelos.items():
    if ruta.is_file():
        tamanos_kb[nombre] = ruta.stat().st_size / 1024
    else:
        tamanos_kb[nombre] = np.nan
        print(f'Advertencia: no se puede medir el tamaño de {nombre}; falta {ruta.name}.')

if not resultados.empty:
    resultados['tamano_kb'] = resultados['modelo'].map(tamanos_kb)
    resultados['tamano_mb'] = resultados['tamano_kb'] / 1024
    display(resultados[['modelo', 'tamano_kb', 'tamano_mb']].round(4))
else:
    print('No hay resultados de modelos a los cuales agregar el tamaño.')

## Criterio 5: Escalabilidad

La **escalabilidad** es la capacidad de mantener un servicio útil cuando crece el volumen de datos, la cantidad de usuarios o el número de solicitudes. No depende solo del algoritmo: también influyen el hardware, el paralelismo, la arquitectura y los procesos de reentrenamiento.

¿El modelo soporta 10× más datos? ¿Se puede reentrenar con facilidad? Vamos a simular una carga de prueba diez veces mayor y medir el tiempo de inferencia por lote y por observación. Es una prueba local sencilla, no una prueba de carga concurrente ni una garantía de desempeño en producción.

In [ ]:
def medir_latencia_lote(modelo, X_datos, repeticiones=3):
    """Mide la latencia media, en milisegundos, de predicciones sobre un lote completo."""
    if repeticiones <= 0:
        raise ValueError('repeticiones debe ser un entero positivo.')
    if len(X_datos) == 0:
        raise ValueError('X_datos no puede estar vacío.')

    # Calentamos el estimador antes de promediar los tiempos de lote.
    modelo.predict(X_datos.iloc[:1])
    duraciones = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        modelo.predict(X_datos)
        duraciones.append(time.perf_counter() - inicio)
    return float(np.mean(duraciones) * 1000)


# Repetimos el conjunto de prueba diez veces y comparamos la duración de los lotes.
if len(X_test_scaled) == 0:
    print('Advertencia: no se simula escalabilidad porque no hay datos de prueba preparados.')
else:
    X_test_10x = pd.concat([X_test_scaled] * 10, ignore_index=True)
    filas_escalabilidad = []
    for nombre, modelo in modelos.items():
        if nombre not in resultados['modelo'].values:
            continue
        tiempo_base_ms = medir_latencia_lote(modelo, X_test_scaled)
        tiempo_10x_ms = medir_latencia_lote(modelo, X_test_10x)
        filas_escalabilidad.append({
            'modelo': nombre,
            'filas_base': len(X_test_scaled),
            'latencia_lote_base_ms': tiempo_base_ms,
            'filas_10x': len(X_test_10x),
            'latencia_lote_10x_ms': tiempo_10x_ms,
            'latencia_por_fila_10x_ms': tiempo_10x_ms / len(X_test_10x),
            'factor_tiempo_lote': tiempo_10x_ms / tiempo_base_ms if tiempo_base_ms else np.nan,
        })
    comparacion_escalabilidad = pd.DataFrame(filas_escalabilidad)
    if comparacion_escalabilidad.empty:
        print('No hay modelos disponibles para la simulación de escalabilidad.')
    else:
        display(comparacion_escalabilidad.round(4))

## Criterio 6: Mantenibilidad

La **mantenibilidad** es la facilidad con que un equipo puede comprender, probar, corregir y actualizar el sistema durante su vida útil. También considera si los datos y el modelo se monitorean, si los cambios quedan registrados y si existe un plan para reentrenar o volver a una versión estable.

¿Tiene tests automatizados? ¿Está documentado? ¿Se registran versiones de datos y modelos? ¿El equipo puede mantenerlo si cambia el personal? ¿Hay alertas ante degradación o fallos? Estas preguntas requieren revisar el proyecto y la operación; no se miden con una celda de código ni quedan resumidas en una métrica del estimador.

Para P2, documenten qué evidencia respalda su evaluación de mantenibilidad y quién será responsable de cada tarea.

## Tabla comparativa final

Ahora integramos todos los criterios en una tabla comparativa. Las categorías cualitativas son una evaluación inicial basada en el tipo de modelo y en las pruebas pequeñas de este notebook; deben validarse para cada aplicación concreta.

In [ ]:
# Integramos métricas medidas y valoraciones cualitativas para los modelos disponibles.
niveles_interpretabilidad = {
    'Random Forest': 'Alta',
    'XGBoost': 'Media',
    'SVM': 'Baja',
}
niveles_escalabilidad = {
    'Random Forest': 'Media',
    'XGBoost': 'Alta',
    'SVM': 'Media',
}
columnas_finales = [
    'modelo', 'accuracy', 'f1', 'latencia_ms', 'tamano_mb',
    'interpretabilidad', 'escalabilidad',
]
if resultados.empty:
    comparacion_final = pd.DataFrame(columns=columnas_finales)
else:
    comparacion_final = resultados.copy()
    comparacion_final['interpretabilidad'] = comparacion_final['modelo'].map(niveles_interpretabilidad)
    comparacion_final['escalabilidad'] = comparacion_final['modelo'].map(niveles_escalabilidad)
    comparacion_final = comparacion_final.reindex(columns=columnas_finales)

ruta_reporte = REPORTS_DIR / 'comparacion_factibilidad.csv'
comparacion_final.to_csv(ruta_reporte, index=False)
display(comparacion_final.round(4))
print(f'Reporte guardado en: {ruta_reporte}')

## Caso de estudio: empresa agrícola

Una empresa agrícola quiere recomendar cultivos con base en nutrientes del suelo y condiciones climáticas. El equipo compara cuatro alternativas: Random Forest, XGBoost, SVM y una Red Neuronal. Los valores que se muestran en la siguiente celda son **hipotéticos**, solo para practicar una decisión con varios criterios; no son resultados del entrenamiento de Crop Recommendation.

La Red Neuronal obtiene accuracy 0.998, latencia 15 ms, tamaño 50 MB e interpretabilidad baja. Consideren también el costo, la facilidad de desplegarla y las consecuencias de equivocarse.

**Pregunta para discusión:** ¿Cuál elegirías y por qué? ¿Qué información adicional necesitarías antes de recomendar su despliegue?

In [ ]:
# Creamos una comparación hipotética para el debate; estos datos no provienen del dataset.
caso_estudio = pd.DataFrame([
    {
        'modelo': 'Random Forest', 'accuracy': 0.995, 'latencia_ms': 8.0,
        'tamano_mb': 1.2, 'interpretabilidad': 'Alta', 'escalabilidad': 'Media',
    },
    {
        'modelo': 'XGBoost', 'accuracy': 0.992, 'latencia_ms': 4.0,
        'tamano_mb': 2.4, 'interpretabilidad': 'Media', 'escalabilidad': 'Alta',
    },
    {
        'modelo': 'SVM', 'accuracy': 0.985, 'latencia_ms': 1.0,
        'tamano_mb': 0.8, 'interpretabilidad': 'Baja', 'escalabilidad': 'Media',
    },
    {
        'modelo': 'Red Neuronal', 'accuracy': 0.998, 'latencia_ms': 15.0,
        'tamano_mb': 50.0, 'interpretabilidad': 'Baja', 'escalabilidad': 'Alta',
    },
])
display(caso_estudio)

## Decisión fundamentada

Una decisión responsable comienza por entender el contexto y el costo de los errores. Después se definen requisitos verificables (por ejemplo, un límite de latencia o un *recall* mínimo), se descartan opciones que no los cumplen y se comparan las alternativas restantes con evidencia. También se consideran los costos de infraestructura y operación, las explicaciones que necesitan las personas usuarias, la escalabilidad y la capacidad del equipo para mantener el sistema.

Antes del despliegue hacen falta pruebas con datos representativos, validación de posibles sesgos, pruebas de carga y un plan de monitoreo y respuesta ante fallos. Una diferencia pequeña en accuracy no justifica automáticamente un modelo mucho más costoso o difícil de explicar.

> **No existe un mejor modelo universal. Depende del contexto.** La decisión debe equilibrar desempeño, riesgos y restricciones reales, y dejar explícitos los supuestos y compromisos aceptados.

## Conclusiones

Antes de llevar un modelo a producción evaluamos seis dimensiones: **rendimiento**, **latencia**, **interpretabilidad**, **costo computacional**, **escalabilidad** y **mantenibilidad**. Ninguna métrica aislada determina la decisión; importa qué errores y restricciones existen en el contexto de uso.

En **P2**, aplicarán estos criterios al modelo de **Wall-Following** y traducirán los errores a costos en dólares. Para ello deberán relacionar las métricas con consecuencias concretas: por ejemplo, cuánto cuesta una detección tardía o una acción equivocada.

**Próxima clase:** métricas de evaluación en escenarios desbalanceados, donde la accuracy puede resultar engañosa.

In [ ]:
# Confirmamos que la clase terminó y recordamos dónde se guardó el reporte generado.
print('Clase 7 completada. Revisa reports/comparacion_factibilidad.csv')